# Lab 8 · The dial that drifts

**Today:** when you walk out you can read a two-level draw, a setting drawn and then data drawn from the setting, and say which loop level is which and what the extra level does to the spread.

**Before you start:** chapter 3 read; lab 7's machines. The loop below has the chapter's two-line shape, a setting drawn and then data drawn from it, with the story removed and different machines on the two lines: a count machine whose dial drifts.

Each section names one idea, explains what it does, and asks you to **predict
what a cell prints before you run it**. Write the prediction down, on paper, out
loud, or in a comment. A prediction you can compare against the output is what
tells you which parts of the code you can already read.

Most sections end with a **Test your understanding** task: write a small piece
of code, then run the check cell under it. The check never grades and never
breaks anything. A ⬜ means not attempted yet, a ❌ means not yet and comes with
a hint, and a ✅ means passing. Run the check cells rather than editing them.
Everything else in the notebook is yours to change.

**AI in this lab.** Until your prediction is written down, work at level 1, with
no AI. The prediction is how you find out what you can read unaided, and both
exams are level 1. Once you have run a cell, level 3 is encouraged: ask your
tutor to explain anything you missed.

Run every cell, and change things to see what happens. Nothing in this notebook
can be broken in a way that matters.

## 1 · One level, then two

A counting machine, dial fixed at 20. Then the same machine, but **each morning the dial itself is drawn**, so some days it truly sits higher and some lower, and the day's count comes from wherever it landed. **Predict: which column of numbers is more spread out, and do the two columns' averages differ?** Commit, then run.

In [ ]:
import numpy as np

rng = np.random.default_rng(61)
fixed_days = []
drifting_days = []
for _ in range(2000):
    fixed_days.append(rng.poisson(20))
    todays_dial = rng.gamma(4, 5)        # averages 4 x 5 = 20
    drifting_days.append(rng.poisson(todays_dial))
print("fixed dial:    mean", round(np.mean(fixed_days), 1),
      " sd", round(np.std(fixed_days), 1))
print("drifting dial: mean", round(np.mean(drifting_days), 1),
      " sd", round(np.std(drifting_days), 1))

Means agree near 20, and the spreads do not: the drifting dial roughly doubles the sd. The two lines inside the loop are the whole idea. Line one draws the setting, and line two draws the data *from that setting*. Averages hide the second level, and **the spread shows it.** Chapter 3 makes the same point with a story attached.

**Test your understanding.** Write `two_level_sd(dial_spread_shape, n_days, rng)`: each day draw `todays_dial = rng.gamma(dial_spread_shape, 20 / dial_spread_shape)` (average stays 20; smaller shape = wilder drift), then a poisson count from it; return the sd of the counts, rounded to 1 decimal.

In [ ]:
# your turn: two_level_sd(dial_spread_shape, n_days, rng)
import numpy as np

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("two_level_sd", expect=10.8, args=(4, 3000, np.random.default_rng(67)))
check("two_level_sd", expect=20.4, args=(1, 3000, np.random.default_rng(67)),
      hint="shape 1 drifts wildly — the sd should come out about double the shape-4 world")

## 2 · Recover the drift from the outside

Now the inversion, chapter 1's grid pointed at chapter 3's dial. You are handed a month of counts from a drifting machine and asked *how wild is the drift?* Strategy: try candidate drift settings, simulate each one, and keep the one whose **spread** matches the observed spread. `argmin`, from lab 7, does the keeping.

**Read first, predict second: which candidate should win, given the month was made with shape 2?**

In [ ]:
import numpy as np

def month_of_counts(shape, n_days, rng):
    counts = []
    for _ in range(n_days):
        todays_dial = rng.gamma(shape, 20 / shape)
        counts.append(rng.poisson(todays_dial))
    return np.array(counts)

rng = np.random.default_rng(71)
observed = month_of_counts(2, 3000, rng)     # the 'data' — shape 2 made it
observed_sd = observed.std()

candidates = np.array([1, 2, 4, 8, 16])
gaps = []
for shape in candidates:
    world = month_of_counts(shape, 3000, rng)
    gaps.append(np.abs(world.std() - observed_sd))
best_position = np.argmin(np.array(gaps))
print("winning candidate:", candidates[best_position])

Shape 2 wins: the machinery recovered the drift setting from spread alone. Every fit in this course is this loop with a different score: candidates, a score per candidate, `argmin`. Chapter 3 scores a candidate by how close its sd comes to the observed one, later chapters use likelihoods, and the loop stays the same.

**Test your understanding.** The score used `np.abs(world.std() - observed_sd)`. Write `match_score(world_sd, observed_sd)` returning that absolute gap, and make it pass its forced check: a perfect match scores 0.

In [ ]:
# your turn: match_score(world_sd, observed_sd)
import numpy as np

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("match_score", expect=0.0, args=(14.2, 14.2))
check("match_score", expect=3.5, args=(10.5, 14.0),
      hint="a gap has no sign — 10.5 vs 14.0 and 14.0 vs 10.5 are the same distance")

## If you finish early

- Rerun section 2 with the observed month made at shape 8. Predict the winner first, and notice which candidates become hard to tell apart (chapter 3's coarse-instrument warning).
- Chapter 3's practice problem 3.1 is this lab at full strength.

## If you are stuck

Wave someone over. This hour exists so that a stuck step costs you a minute
rather than an evening. Known snags:

- **`two_level_sd` misses by a lot.** Check the scale: `20 / dial_spread_shape` keeps the average at 20, and a fixed scale of 5 does not.
- **Section 2 picks the wrong candidate on your edit.** Spreads from 3000-day months still wobble, and nearby candidates genuinely overlap. The overlap is a finding rather than a bug.
- **`match_score` fails the second check.** Use `np.abs` rather than a bare subtraction.